# Fragments testing for CMS2

This notebook must be run within the oktoberfest conda env from ubuntu subsystem. It is intended to test adding modifcations for crosslinking

In [2]:
import spectrum_fundamentals.constants as constants
import spectrum_fundamentals.fragments as fragments
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
sequence = "AK[UNIMOD:1886]"
mass_analyzer = "FTMS"
charge = 2
df = fragments.initialize_peaks(sequence, mass_analyzer, charge)
print(df)
#df[0].sort_values("ion_type")

(  ion_type  no  charge        mass    min_mass    max_mass
2        b   1       2   36.525833   36.525103   36.526564
0        b   1       1   72.044390   72.042950   72.045831
3        y   1       2  116.586422  116.584091  116.588754
6        b   2       2  143.099697  143.096835  143.102559
7        y   2       2  152.104979  152.101937  152.108021
1        y   1       1  232.165568  232.160925  232.170211
4        b   2       1  285.192117  285.186414  285.197821
5        y   2       1  303.202682  303.196618  303.208746, 1, 'AK', 302.19540567)


In [3]:
from typing import Dict, List, Optional, Tuple
import pandas as pd
import re

def initialize_peaks_xl(
    sequence: str, mass_analyzer: str, crosslinker_position: int, crosslinker_type: str
) -> Tuple[pd.DataFrame, int, str, float, float]:
    """Generate theoretical peaks for a modified (potentially cleavable cross-linked) peptide sequence.

    This function get only one modified peptide

    :param sequence: Modified peptide sequence
    :param mass_analyzer: Type of mass analyzer used eg. FTMS, ITMS
    :param crosslinker_position: the position of crosslinker connected to lysine
    :param crosslinker_type: Can be either DSSO, DSBU or BuUrBU
    :raises ValueError: if crosslinker_type be unkown
    :return: List of theoretical peaks
    """
    charge = 2  # generate only peaks with charge 1 and 2
    crosslinker_type = crosslinker_type.upper()
    if crosslinker_type == "DSSO":
        dsso = "[UNIMOD:1896]"
        dsso_s = "[UNIMOD:1881]"
        dsso_l = "[UNIMOD:1882]"
        sequence_s = sequence.replace(dsso, dsso_s)
        sequence_l = sequence.replace(dsso, dsso_l)
    elif crosslinker_type in ["DSBU", "BUURBU"]:
        dsbu = "[UNIMOD:1884]"
        dsbu_s = "[UNIMOD:1886]"
        dsbu_l = "[UNIMOD:1885]"
        sequence_s = sequence.replace(dsbu, dsbu_s)
        sequence_l = sequence.replace(dsbu, dsbu_l)
    else:
        raise ValueError(f"Unkown crosslinker type: {crosslinker_type}")

    df_out_s, tmt_n_term, peptide_sequence, calc_mass_s = fragments.initialize_peaks(sequence_s, mass_analyzer, charge)
    df_out_l, tmt_n_term, peptide_sequence, calc_mass_l = fragments.initialize_peaks(sequence_l, mass_analyzer, charge)

    threshold_b = crosslinker_position
    threshold_y = len(peptide_sequence) - crosslinker_position + 1

    df_out_s.loc[(df_out_s["no"] >= threshold_b) & (df_out_s["ion_type"] == "b"), "ion_type"] = "b-short"
    df_out_s.loc[(df_out_s["no"] >= threshold_y) & (df_out_s["ion_type"] == "y"), "ion_type"] = "y-short"
    df_out_l.loc[(df_out_l["no"] >= threshold_b) & (df_out_l["ion_type"] == "b"), "ion_type"] = "b-long"
    df_out_l.loc[(df_out_l["no"] >= threshold_y) & (df_out_l["ion_type"] == "y"), "ion_type"] = "y-long"

    concatenated_df = pd.concat([df_out_s, df_out_l])
    unique_df = concatenated_df.drop_duplicates()
    df_out = unique_df.sort_values("mass")

    return df_out, tmt_n_term, peptide_sequence, calc_mass_s, calc_mass_l

In [4]:
initialize_peaks_xl("K[UNIMOD:1884]K", "FTMS", 1, crosslinker_type="BUUrBU")


NameError: name 'fragments' is not defined

In [9]:
import re
import pandas as pd

sequence = "AK[UNIMOD:1896]KKKKKK"
mass_analyzer = "FTMS"
kind_crosslinker = "DO"
crosslinker_position = 2
charge = 2  #  generate only peaks with charge 1 and 2 

#df = fragments.initialize_peaks(sequence, mass_analyzer, charge)

if kind_crosslinker == "DSSO":
    sequence_s = re.sub("\[[^]]*\]", lambda x:x.group(0).replace('UNIMOD:1896','UNIMOD:1881'), sequence)
    sequence_l = re.sub("\[[^]]*\]", lambda x:x.group(0).replace('UNIMOD:1896','UNIMOD:1882'), sequence)
elif kind_crosslinker == "DSBU" or "BuUrBU":
    sequence_s = re.sub("\[[^]]*\]", lambda x:x.group(0).replace('UNIMOD:1884','UNIMOD:1886'), sequence)
    sequence_l = re.sub("\[[^]]*\]", lambda x:x.group(0).replace('UNIMOD:1884','UNIMOD:1885'), sequence)    
else:
    raise Exception("Sorry, This crosslinker is not supported by xl_prosit")
    
  

df_out_s = fragments.initialize_peaks(sequence_s, mass_analyzer, charge)
df_out_l = fragments.initialize_peaks(sequence_l, mass_analyzer, charge)


peptide_sequence = re.sub(r"\[.*?\]", "", sequence)


threshold_b = crosslinker_position
threshold_y = len(peptide_sequence) - crosslinker_position + 1

df_out_s[0].loc[(df_out_s[0]['no'] >= threshold_b) & (df_out_s[0]['ion_type'] == 'b'), 'ion_type'] = 'b-short'
df_out_s[0].loc[(df_out_s[0]['no'] >= threshold_y) & (df_out_s[0]['ion_type'] == 'y'), 'ion_type'] = 'y-short'
df_out_l[0].loc[(df_out_l[0]['no'] >= threshold_b) & (df_out_l[0]['ion_type'] == 'b'), 'ion_type'] = 'b-long'
df_out_l[0].loc[(df_out_l[0]['no'] >= threshold_y) & (df_out_l[0]['ion_type'] == 'y'), 'ion_type'] = 'y-long'


concatenated_df = pd.concat([df_out_s[0], df_out_l[0]])
unique_df = concatenated_df.drop_duplicates()
"""
print(df_out_s[0])
print("next")
print(df_out_l[0])
print("next")
print(concatenated_df)
print("next")
"""
df_out = unique_df.sort_values("mass") 
#print(unique_df.sort_values("mass"))
print(df_out)



   ion_type  no  charge         mass     min_mass     max_mass
2         b   1       2    36.525833    36.525103    36.526564
0         b   1       1    72.044390    72.042950    72.045831
3         y   1       2    74.060040    74.058559    74.061522
7         y   2       2   138.107522   138.104760   138.110284
1         y   1       1   147.112804   147.109862   147.115746
6    b-long   2       2   179.575197   179.571606   179.578789
6   b-short   2       2   179.575197   179.571606   179.578789
11        y   3       2   202.155003   202.150960   202.159046
10  b-short   3       2   243.622679   243.617807   243.627551
10   b-long   3       2   243.622679   243.617807   243.627551
15        y   4       2   266.202485   266.197161   266.207809
5         y   2       1   275.207767   275.202263   275.213271
14  b-short   4       2   307.670160   307.664007   307.676314
14   b-long   4       2   307.670160   307.664007   307.676314
19        y   5       2   330.249966   330.243361   330

In [17]:
sequence = "K[UNIMOD:1896]M[UNIMOD:35]R"
mass_analyzer = "FTMS"
crosslinker_position = 3
crosslinker_type = "dsso"
df = fragments.initialize_peaks_xl(sequence, mass_analyzer, crosslinker_position, crosslinker_type )
display(df)
#df[0].sort_values("ion_type")

(   ion_type  no  charge        mass    min_mass    max_mass
 3   y-short   1       2   88.063114   88.061353   88.064876
 3    y-long   1       2   88.063114   88.061353   88.064876
 2         b   1       2   92.060040   92.058199   92.061882
 2         b   1       2  108.046075  108.043915  108.048236
 7   y-short   2       2  161.580814  161.577582  161.584046
 7    y-long   2       2  161.580814  161.577582  161.584046
 6         b   2       2  165.577740  165.574429  165.581052
 1   y-short   1       1  175.118952  175.115450  175.122455
 1    y-long   1       1  175.118952  175.115450  175.122455
 6         b   2       2  181.563775  181.560144  181.567407
 0         b   1       1  183.112804  183.109142  183.116467
 0         b   1       1  215.084874  215.080573  215.089176
 10  b-short   3       2  243.628296  243.623423  243.633168
 11  y-short   3       2  252.633578  252.628525  252.638631
 10   b-long   3       2  259.614331  259.609138  259.619523
 11   y-long   3       2

In [16]:
sequence = "AK[UNIMOD:1896]M[UNIMOD:35]R"
mass_analyzer = "FTMS"
charge = 2
df = fragments.initialize_peaks(sequence, mass_analyzer, charge)
print(df)
#df[0].sort_values("ion_type")

(Empty DataFrame
Columns: [ion_type, no, charge, mass, min_mass, max_mass]
Index: [], -1, '', 0.0)
